## Util

In [1]:
%load_ext autoreload
%autoreload 2

## Data preprocessing

In [2]:
import sys

from data_loader import load_scam_data, load_embeddings

dataset = load_scam_data()

/home/minhngo/miniconda3/envs/cs3264-project-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloading/Loading SMS Spam dataset...



Sample Data Structure:
{'sms': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n', 'label': 0}


### Tokenize and build vocabulary

In [3]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    # This handles truncation and padding automatically
    return tokenizer(examples["sms"], truncation=True, padding="max_length", max_length=128)

# Map the tokenizer over your existing Hugging Face dataset
tokenized_sms = dataset.map(preprocess_function, batched=True)

In [4]:
print(tokenized_sms["train"].features)

{'sms': Value('string'), 'label': ClassLabel(names=['ham', 'spam']), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


## Core model

In [5]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=2 # Ham vs Scam
)

Loading weights: 100%|█████████████| 100/100 [00:00<00:00, 377.22it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Training

In [9]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5, # Transformers need very small learning rates
    per_device_train_batch_size=16,
    num_train_epochs=3, # Transformers converge very fast
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

split_datasets = tokenized_sms["train"].train_test_split(test_size=0.2, seed=42)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_datasets["train"],
    eval_dataset=split_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/home/minhngo/miniconda3/envs/cs3264-project-env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Evaluation

In [ ]:
for log in trainer.state.log_history:
    print(log)

metrics = trainer.evaluate()
print(metrics)

## Statistics Visualization

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_loss = [x['loss'] for x in history if 'loss' in x]
val_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

plt.plot(train_loss, label='Train Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend()
plt.show()